In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *
from sklearn.metrics import mean_absolute_error, mean_squared_error
from models import *
from plots import *

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
data = load_data()

In [ ]:
target = 't_seasdiff'
exog=['lagged_tmed', 'lagged_prec', 'lagged_tmin', 'lagged_tmax']

In [ ]:
data.head()

In [ ]:
if target == 't_seasdiff':
    data["t_seasdiff"]= data["tdiff"].diff(12)
    data = data.dropna(subset=["t_seasdiff"])

# SARIMA / SARIMAX

In [ ]:
len(data)

In [ ]:
p, d, q = 0, 1, 1
P, D, Q, s = 1, 0, 0, 12


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

def _plot_cv_results(df, target, split_metrics, forecasts, actuals, 
                     train_months, forecast_months):
    """Helper function to visualize cross-validation results."""
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Plot 1: All data with CV splits highlighted
    ax1 = axes[0]
    ax1.plot(df.index, df[target], label='Full Data', alpha=0.7, color='blue')
    
    for i, split_info in enumerate(split_metrics):
        test_start = split_info['test_start']
        test_end = split_info['test_end']
        ax1.axvspan(test_start, test_end, alpha=0.2, color='red', 
                   label='Test Period' if i == 0 else '')
        ax1.axvline(split_info['train_end'], color='green', linestyle='--', 
                   alpha=0.5, label='Train End' if i == 0 else '')
    
    ax1.set_title(f'Time Series with CV Splits ({train_months//12}yr train, {forecast_months}mo forecast)')
    ax1.set_xlabel('Date')
    ax1.set_ylabel(target)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Forecast vs Actual for test periods
    ax2 = axes[1]
    
    # Reconstruct index for forecasts
    forecast_indices = []
    for split_info in split_metrics:
        test_start = split_info['test_start']
        test_end = split_info['test_end']
        indices = df.loc[test_start:test_end].index
        forecast_indices.extend(indices[:forecast_months])
    
    ax2.plot(forecast_indices, actuals, label='Actual', marker='o', 
            markersize=3, linewidth=1.5)
    ax2.plot(forecast_indices, forecasts, label='Forecast', marker='x', 
            markersize=3, linewidth=1.5, alpha=0.7)
    
    ax2.set_title('Forecast vs Actual (SARIMAX)')
    ax2.set_xlabel('Date')
    ax2.set_ylabel(target)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Plot metrics over splits
    metrics_df = pd.DataFrame(split_metrics)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(metrics_df['split'], metrics_df['rmse'], marker='o', label='RMSE')
    ax.plot(metrics_df['split'], metrics_df['mae'], marker='s', label='MAE')
    ax.set_xlabel('Split Number')
    ax.set_ylabel('Error')
    ax.set_title('Forecast Errors Across CV Splits')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def sarimax_time_series_cv(df, target, p, d, q, P, D, Q, s, 
                           train_years=9, forecast_months=12, 
                           test_start_year=2010, exog=None):
    """
    Perform time series cross-validation for SARIMAX models.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with datetime index
    target : str
        Name of target column
    p, d, q : int
        Non-seasonal ARIMA parameters
    P, D, Q, s : int
        Seasonal ARIMA parameters
    train_years : int
        Number of years to use for training (default: 9)
    forecast_months : int
        Number of months to forecast ahead (default: 12)
    test_start_year : int
        Year to start testing from (default: 2010)
    exog : list of str, optional
        Names of exogenous variables
        
    Returns:
    --------
    dict : Dictionary containing results and metrics
    """
    
    # Calculate number of splits based on available test data
    test_data = df[df.index.year >= test_start_year]
    total_test_months = len(test_data)
    train_months = train_years * 12
    
    # Calculate splits: we can make a prediction every forecast_months
    n_splits = (total_test_months - train_months) // forecast_months
    
    print(f"Data range: {df.index.min().year} to {df.index.max().year}")
    print(f"Training window: {train_years} years ({train_months} months)")
    print(f"Forecast horizon: {forecast_months} months")
    print(f"Testing period: {test_start_year} onwards")
    print(f"Number of CV splits: {n_splits}\n")
    
    # Initialize storage for results
    all_forecasts = []
    all_actuals = []
    all_train_ends = []
    split_metrics = []
    
    # Perform time series cross-validation
    for split_idx in range(n_splits):
        # Calculate train/test indices
        train_start_idx = (test_start_year - df.index.min().year) * 12 + split_idx * forecast_months
        train_end_idx = train_start_idx + train_months
        test_end_idx = train_end_idx + forecast_months
        
        # Extract train and test data
        train = df[target].iloc[train_start_idx:train_end_idx]
        test = df[target].iloc[train_end_idx:test_end_idx]
        
        # Handle exogenous variables
        exog_train = exog_test = None
        if exog is not None:
            exog_train = df[exog].iloc[train_start_idx:train_end_idx]
            exog_test = df[exog].iloc[train_end_idx:test_end_idx]
        
        # Fit model
        try:
            model = SARIMAX(train, exog=exog_train, 
                          order=(p, d, q), 
                          seasonal_order=(P, D, Q, s))
            results = model.fit(disp=False)
            
            # Forecast
            forecast = results.forecast(steps=forecast_months, exog=exog_test)
            
            # Calculate metrics for this split
            rmse = np.sqrt(mean_squared_error(test, forecast))
            mae = mean_absolute_error(test, forecast)
            
            # Store results
            all_forecasts.extend(forecast.values)
            all_actuals.extend(test.values)
            all_train_ends.append(train.index[-1])
            split_metrics.append({
                'split': split_idx + 1,
                'train_end': train.index[-1],
                'test_start': test.index[0],
                'test_end': test.index[-1],
                'rmse': rmse,
                'mae': mae
            })
            
            print(f"Split {split_idx + 1}/{n_splits} - "
                  f"Train: {train.index[0].strftime('%Y-%m')} to {train.index[-1].strftime('%Y-%m')}, "
                  f"Test: {test.index[0].strftime('%Y-%m')} to {test.index[-1].strftime('%Y-%m')} - "
                  f"RMSE: {rmse:.4f}, MAE: {mae:.4f}")
            
        except Exception as e:
            print(f"Split {split_idx + 1} failed: {str(e)}")
            continue
    
    # Calculate overall metrics
    overall_rmse = np.sqrt(mean_squared_error(all_actuals, all_forecasts))
    overall_mae = mean_absolute_error(all_actuals, all_forecasts)
    avg_rmse = np.mean([m['rmse'] for m in split_metrics])
    avg_mae = np.mean([m['mae'] for m in split_metrics])
    
    print(f"\n{'='*70}")
    print(f"OVERALL METRICS (concatenated predictions):")
    print(f"RMSE: {overall_rmse:.4f}")
    print(f"MAE: {overall_mae:.4f}")
    print(f"\nAVERAGE METRICS (across splits):")
    print(f"Average RMSE: {avg_rmse:.4f}")
    print(f"Average MAE: {avg_mae:.4f}")
    print(f"{'='*70}\n")
    
    # Visualize results
    _plot_cv_results(df, target, split_metrics, all_forecasts, all_actuals, 
                     train_months, forecast_months)
    
    return {
        'split_metrics': pd.DataFrame(split_metrics),
        'overall_rmse': overall_rmse,
        'overall_mae': overall_mae,
        'avg_rmse': avg_rmse,
        'avg_mae': avg_mae,
        'forecasts': all_forecasts,
        'actuals': all_actuals
    }


In [ ]:
results = sarimax_time_series_cv(data, 'tdiff', p, d, q, P, D, Q, s, test_start_year=1995, exog=['lagged_prec', 'lagged_tmin', 'lagged_tmax', 'lagged_tmed'])

In [ ]:
rmse, mae, results, forecast, test, train = sarimax_experiment(data, target, p, d, q, P, D, Q, s)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=(p,d,q),
    seasonal_order=(P,D,Q,12),
    rmse=rmse,
    mae=mae,
)

In [ ]:
rmse, mae, results, forecast, test, train = sarimax_experiment(data, target, p, d, q, P, D, Q, s, exog= exog)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=(p,d,q),
    seasonal_order=(P,D,Q,12),
    rmse=rmse,
    mae=mae,
)

In [ ]:
from utils import sarima_candidates_from_acf

cand_res = sarima_candidates_from_acf(data[target], m=12, max_p=3, max_q=2, max_P=3, max_Q=3, max_lag=50)

candidates = cand_res["candidates"]
for i, (p, d, q, P, D, Q, s) in enumerate(candidates, 1):
    print(f"{i:02d}. SARIMA({p},{d},{q})({P},{D},{Q},{s})")

In [ ]:
best_info, scores_df = sarimax_grid_search(
    df=data,
    target=target,
    candidates=candidates,
    forecast_window=12,
    no_windows=10,
    exog=None,    # or list of exogenous column names
    metric="rmse",
    verbose=True,
)

In [ ]:
plot_sarimax_results(
    train=best_info["train"],
    test=best_info["test"],
    forecast=best_info["forecast"],
    results=best_info["results"],
    order=best_info["order"],
    seasonal_order=best_info["seasonal_order"],
    rmse=best_info["rmse"],
    mae=best_info["mae"],
)

In [ ]:
best_info, scores_df = sarimax_grid_search(
    df=data,
    target=target,
    candidates=candidates,
    forecast_window=12,
    no_windows=10,
    exog=None,    
    metric="rmse",
    verbose=True,
)

In [ ]:
plot_sarimax_results(
    train=train,
    test=best_info["test"],
    forecast=best_info["forecast"],
    results=best_info["results"],
    order=best_info["order"],
    seasonal_order=best_info["seasonal_order"],
    rmse=best_info["rmse"],
    mae=best_info["mae"],
)

In [ ]:
best_info, scores_df = sarimax_grid_search(
    df=data,
    target=target,
    candidates=candidates,
    forecast_window=12,
    no_windows=10,
    exog=exog,    # or list of exogenous column names
    metric="rmse",
    verbose=True,
)

In [ ]:
plot_sarimax_results(
    train=best_info["train"],
    test=best_info["test"],
    forecast=best_info["forecast"],
    results=best_info["results"],
    order=best_info["order"],
    seasonal_order=best_info["seasonal_order"],
    rmse=best_info["rmse"],
    mae=best_info["mae"],
)

## Using AutoSarima with the current train/test split

In [ ]:
rmse, mae, results, forecast, test, train, order, seasonal_order = auto_sarima_experiment(data, target, forecast_window=12, no_windows=10, m=12)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=order,
    seasonal_order=seasonal_order,
    rmse=rmse,
    mae=mae,
)

In [ ]:
rmse, mae, results, forecast, test, train, order, seasonal_order = auto_sarima_experiment(data, target, forecast_window=12, no_windows=10, m=12, exog=exog)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=order,
    seasonal_order=seasonal_order,
    rmse=rmse,
    mae=mae,
)

In [ ]:
h = 12
years_test = 2
years_train = 8
forecast_window = h * years_test
step = h
train_size = h * years_train
best_info, scores_df  = select_best_sarima_cv(
    data,
    target,
    candidates=candidates,
    forecast_window=forecast_window,
    train_size=train_size,
    step=step,
    exog=None,
)

In [ ]:
p, d, q = best_info["order"][0] , best_info["order"][1], best_info["order"][2]
P, D, Q, s = best_info["seasonal_order"][0], best_info["seasonal_order"][1], best_info["seasonal_order"][2], best_info["seasonal_order"][3]
rmse, mae, results_train, forecast, test, train = sarimax_experiment(data, target, p, d, q, P, D, Q, s)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results_train,     # ✅ use results_train
    order=(p, d, q),
    seasonal_order=(P, D, Q, s),
    rmse=rmse,
    mae=mae,
)

In [ ]:
best_info, scores_df  = select_best_sarima_cv(
    data,
    target,
    candidates=candidates,
    forecast_window=forecast_window,
    train_size=train_size,
    step=step,
    exog=exog,
)

In [ ]:
p, d, q = best_info["order"][0] , best_info["order"][1], best_info["order"][2]
P, D, Q, s = best_info["seasonal_order"][0], best_info["seasonal_order"][1], best_info["seasonal_order"][2], best_info["seasonal_order"][3]
rmse, mae, results_train, forecast, test, train = sarimax_experiment(data, target, p, d, q, P, D, Q, s)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=order,
    seasonal_order=seasonal_order,
    rmse=rmse,
    mae=mae,
)

# LSTMs

In [ ]:
res = lstm_grid_search_cv(data, 'tdiff', train_years=8, forecast_months=24, test_start_year=1996, epochs=60, verbose=0, param_grid={
    'exog': [['lagged_tmed', 'lagged_tmin', 'lagged_prec', 'lagged_tmax'], ['lagged_tmin', 'lagged_prec', 'lagged_tmax'], ['lagged_tmed', 'lagged_prec'], ['lagged_tmin', 'lagged_prec', 'lagged_tmax']],
    'lr': [0.001],
    'hidden_size': [50, 25, 100],
    'num_layers': [1, 2],
    'dropout': [0.1, 0.2, 0.3]
    
})

In [ ]:
print_grid_search_results(res['results'], res['best_overall_score'], res['best_overall_results'], res['best_overall_params'],res['best_avg_score'], res['best_avg_results'], res['best_avg_params'],res['best_last_fold_score'], res['best_last_fold_results'], res['best_last_fold_params'])

In [ ]:
results = lstm_time_series_cv(
    df=data,
    target='tdiff',
    train_years=8,
    forecast_months=24,
    test_start_year=1996,
    exog=['lagged_tmin', 'lagged_prec', 'lagged_tmax', 'lagged_tdiff'],  # optional
    seq_length=24,
    epochs=60,
    batch_size=32,
    lr=0.001,
    hidden_size=100,
    num_layers=2,
    dropout=0.2,
    plot=True,
)

In [ ]:
results = lstm_time_series_cv(
    df=data,
    target='tdiff',
    train_years=8,
    forecast_months=24,
    test_start_year=1996,
    exog=['lagged_tmed', 'lagged_tmin', 'lagged_prec', 'lagged_tmax', 'lagged_tdiff'],  # optional
    seq_length=24,
    epochs=60,
    batch_size=32,
    lr=0.001,
    hidden_size=50,
    num_layers=2,
    dropout=0.2,
    plot=True,
)